# 05 — Barpeta weekly rebuild (May→Oct)

Generates weekly transparent flood masks and seeded evacuation scenarios into `docs/scenarios/`.

In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox

if not Path('src').exists() and Path('../src').exists():
    os.chdir('..')

from src import io, raster_features, simulation, scenario_export
from src.config import WET_CLASS_VALUES, WEEKLY_START_DATE, WEEKLY_END_DATE, DEFAULT_SCENARIO_SEEDS, DEFAULT_ACCESS_RADIUS_M

DOCS_ROOT = Path('docs')
SCENARIOS_ROOT = DOCS_ROOT / 'scenarios'
RESET_SCENARIOS_DIR = True

START_DATE = WEEKLY_START_DATE
END_DATE = WEEKLY_END_DATE
MASK_DEM_WEIGHT = 0.25
MASK_THRESHOLD = 0.50
MASK_RGBA = (65, 155, 223)
MASK_ALPHA = 150

SCENARIO_SEEDS = list(DEFAULT_SCENARIO_SEEDS)
DAMAGE_MONTH = 'oct'
N_TRUCKS = 20
ACCESS_RADIUS_M = float(DEFAULT_ACCESS_RADIUS_M)
WATER_OVER_THRESHOLD = 0.95
P_BACKGROUND = 0.02
ROUTE_WEIGHT = 'length'
EXIT_K = 24
EXIT_EPSILON_DEG = 0.003

GRAPHML_PATH = Path('outputs/tables/roads_drive_lcc.graphml')
ROADS_FEATURES_PARQUET = Path('outputs/tables/roads_edges_features.parquet')
ROADS_FEATURES_GPKG = Path('outputs/tables/roads_edges_features.gpkg')
BUILDINGS_SNAPSHOT_PATH = Path('outputs/tables/buildings_features_snapshot.parquet')

weekly_dates = pd.date_range(START_DATE, END_DATE, freq='7D')
if weekly_dates[-1] != pd.Timestamp(END_DATE):
    weekly_dates = weekly_dates.append(pd.DatetimeIndex([pd.Timestamp(END_DATE)]))
n_frames = len(weekly_dates)

aoi = io.load_aoi_full().to_crs('EPSG:4326')
buildings = gpd.read_parquet(BUILDINGS_SNAPSHOT_PATH) if BUILDINGS_SNAPSHOT_PATH.exists() else io.load_buildings()
if buildings.crs is None:
    buildings = buildings.set_crs(aoi.crs)
pop_features = io.load_population_features()

with io.open_lulc_month('may') as src_may, io.open_lulc_month('oct') as src_oct, io.open_dem() as src_dem:
    weekly = raster_features.build_weekly_wet_masks_from_may_oct_dem(
        src_may, src_oct, src_dem,
        n_frames=n_frames, wet_classes=WET_CLASS_VALUES,
        dem_weight=MASK_DEM_WEIGHT, threshold=MASK_THRESHOLD
    )
masks = weekly['masks']

G = ox.load_graphml(GRAPHML_PATH)
if ROADS_FEATURES_PARQUET.exists():
    edges_features = gpd.read_parquet(ROADS_FEATURES_PARQUET)
elif ROADS_FEATURES_GPKG.exists():
    edges_features = gpd.read_file(ROADS_FEATURES_GPKG)
else:
    raise FileNotFoundError('Missing roads_edges_features parquet/gpkg')
G = simulation.attach_edge_features_from_gdf(G, edges_features, strict=False)

bounds = tuple(aoi.total_bounds.tolist())
cent = aoi.unary_union.centroid
source_node = ox.distance.nearest_nodes(G, X=float(cent.x), Y=float(cent.y))
exit_nodes = simulation.choose_exit_nodes_boundary_bbox(G, bbox=bounds, k=EXIT_K, epsilon_deg=EXIT_EPSILON_DEG)
if not exit_nodes:
    raise RuntimeError('No exit nodes found; increase EXIT_EPSILON_DEG')

with io.open_population_raster() as pop_src:
    buildings_people, pop_method = simulation.allocate_population_hybrid(
        buildings, pop_features=pop_features, pop_raster_src=pop_src,
        out_col='people', building_id_col='building_id'
    )

total_people = int(buildings_people['people'].sum())
if RESET_SCENARIOS_DIR:
    scenario_export.reset_dir(SCENARIOS_ROOT)
else:
    scenario_export.ensure_dir(SCENARIOS_ROOT)

index_path = SCENARIOS_ROOT / 'index.json'
summary_rows = []

for seed in SCENARIO_SEEDS:
    scenario_id = f'barpeta_{DAMAGE_MONTH}_seed_{seed}'
    scenario_dir = SCENARIOS_ROOT / scenario_id
    scenario_export.reset_dir(scenario_dir)

    scenario_export.write_weekly_frame_stack(masks, scenario_dir/'frames'/'water_mask', rgb=MASK_RGBA, alpha=MASK_ALPHA)

    params = simulation.SimulationParams(
        month=DAMAGE_MONTH, n_trucks=N_TRUCKS, n_people=total_people,
        access_radius_m=ACCESS_RADIUS_M, seed=int(seed), route_weight=ROUTE_WEIGHT,
        failure=simulation.FailureModelParams(water_over_threshold=WATER_OVER_THRESHOLD, p_background=P_BACKGROUND),
    )

    detail = simulation.run_one_realization_detailed(
        G=G, buildings=buildings_people, source_node=source_node, exit_nodes=exit_nodes,
        params=params, rng=np.random.default_rng(seed), people_col='people'
    )

    metrics = dict(detail['metrics'])
    metrics.update({
        'scenario_id': scenario_id, 'seed': int(seed), 'allocation_method': pop_method,
        'n_people_allocated': total_people, 'n_frames': int(n_frames),
        'date_start': str(pd.Timestamp(START_DATE).date()), 'date_end': str(pd.Timestamp(END_DATE).date()),
    })

    scenario_export.write_json(scenario_dir/'metrics.json', metrics)
    scenario_export.write_csv(scenario_dir/'failed_edges.csv', detail['failed_edges'])
    scenario_export.write_csv(scenario_dir/'evac_edges.csv', detail['evac_edges'])
    scenario_export.write_json(
        scenario_dir/'routes.json',
        {
            'source_node': str(source_node),
            'exit_nodes': [str(e) for e in exit_nodes],
            'routes': [[str(n) for n in r] if r is not None else None for r in detail['routes']],
        },
    )

    b_access = detail['buildings'].copy()
    if 'geometry' in b_access.columns:
        b_access['geometry_wkt'] = b_access.geometry.to_wkt()
        b_access = b_access.drop(columns=['geometry'])
    scenario_export.write_csv(scenario_dir/'buildings_access.csv', b_access)

    manifest = scenario_export.build_manifest(
        scenario_id=scenario_id, n_frames=n_frames,
        start_date=str(pd.Timestamp(START_DATE).date()), end_date=str(pd.Timestamp(END_DATE).date()),
        parameters={
            'seed': int(seed), 'damage_month': DAMAGE_MONTH, 'access_radius_m': ACCESS_RADIUS_M,
            'water_over_threshold': WATER_OVER_THRESHOLD, 'p_background': P_BACKGROUND,
            'mask_dem_weight': MASK_DEM_WEIGHT, 'mask_threshold': MASK_THRESHOLD,
            'population_allocation_method': pop_method,
        },
        assets={
            'metrics': 'metrics.json', 'failed_edges': 'failed_edges.csv',
            'evac_edges': 'evac_edges.csv', 'routes': 'routes.json', 'buildings_access': 'buildings_access.csv',
        },
        aoi_bounds_wgs84=list(bounds),
    )
    scenario_export.write_json(scenario_dir/'manifest.json', manifest)

    scenario_export.update_index_json(
        index_path,
        {
            'scenario_id': scenario_id,
            'path': f'scenarios/{scenario_id}/manifest.json',
            'parameters': {'seed': int(seed), 'damage_month': DAMAGE_MONTH, 'access_radius_m': ACCESS_RADIUS_M},
        },
        options_updates={'seed': [int(seed)], 'damage_month': [DAMAGE_MONTH], 'access_radius_m': [ACCESS_RADIUS_M]},
    )

    summary_rows.append(metrics)

summary = pd.DataFrame(summary_rows).sort_values('seed').reset_index(drop=True)
summary.to_csv(SCENARIOS_ROOT/'scenario_summary.csv', index=False)
print('Exported to', SCENARIOS_ROOT.resolve())
display(summary[['scenario_id','seed','n_failed_edges','frac_routes_found','access_frac','mean_dist','median_dist']])

C:\Users\nachi\AppData\Local\Temp\ipykernel_215020\1682238573.py:69: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  cent = aoi.unary_union.centroid
c:\Users\nachi\Data\GitHubProjects\Barpeta flooding\barpeta_evac\src\simulation.py:627: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  inter["_iarea"] = inter.geometry.area.astype(float)


Exported to C:\Users\nachi\Data\GitHubProjects\Barpeta flooding\barpeta_evac\docs\scenarios


,scenario_id,seed,n_failed_edges,frac_routes_found,access_frac,mean_dist,median_dist
0,barpeta_oct_seed_7,7,58,0.90,0.45,1582.564869,1334.825215
1,barpeta_oct_seed_42,42,65,0.85,0.46,2803.593461,1334.825215
2,barpeta_oct_seed_101,101,44,0.90,0.39,2547.149094,1705.235878
3,barpeta_oct_seed_202,202,60,1.00,0.61,1054.845719,618.784313
4,barpeta_oct_seed_404,404,69,1.00,0.40,1724.896531,1646.915886
